# Case Study Data Acquisition

Ingest SpaceX launch and rocket data from APIs, clean and merge the data, and store it in a
local SQLite3 database. Use SQL queries to analyze the launch data.
api.spacexdata.com/v4/launches
api.spacexdata.com/v4/rockets
You are tasked with analyzing real-time data from the SpaceX API and enhancing it with
simulated attributes to understand how real-world datasets are built and analyzed using Python
and SQL.

## Step 1: Load SpaceX Launch Data from API
● Use https://api.spacexdata.com/v4/launches

● Extract relevant columns: name, date_utc, success, details, rocket

● Convert date_utc to datetime and extract the year

In [119]:
import requests
import pandas as pd

In [120]:
#1: Fetch data from SpaceX API
url = "https://api.spacexdata.com/v4/launches"
response = requests.get(url)

# Convert JSON to Python list
data = response.json()

print(f"Total launches: {len(data)}")

Total launches: 205


In [121]:
#2: Convert to DataFrame
df = pd.DataFrame(data)

In [122]:
#3: Select required columns
df = df[["name", "date_utc", "success", "details", "rocket"]]

In [124]:
#4: Convert date_utc to datetime
df["date_utc"] = pd.to_datetime(df["date_utc"])

In [125]:
#5: Extract year
df["year"] = df["date_utc"].dt.year

In [126]:
# Preview data
print(df.head(10))

                   name                  date_utc success  \
0             FalconSat 2006-03-24 22:30:00+00:00   False   
1               DemoSat 2007-03-21 01:10:00+00:00   False   
2           Trailblazer 2008-08-03 03:34:00+00:00   False   
3                RatSat 2008-09-28 23:15:00+00:00    True   
4              RazakSat 2009-07-13 03:35:00+00:00    True   
5  Falcon 9 Test Flight 2010-06-04 18:45:00+00:00    True   
6                COTS 1 2010-12-08 15:43:00+00:00    True   
7                COTS 2 2012-05-22 07:44:00+00:00    True   
8                 CRS-1 2012-10-08 00:35:00+00:00    True   
9                 CRS-2 2013-03-01 19:10:00+00:00    True   

                                             details  \
0   Engine failure at 33 seconds and loss of vehicle   
1  Successful first stage burn and transition to ...   
2  Residual stage 1 thrust led to collision betwe...   
3  Ratsat was carried to orbit on the first succe...   
4                                               

## Step 2: Load Rocket Metadata
● Use  https://api.spacexdata.com/v4/rockets

● Extract id, name, type, active, and stages

In [127]:
#1: Fetch SpaceX rocket metadata from API
url = "https://api.spacexdata.com/v4/rockets"
response = requests.get(url)

# Convert JSON to Python list
rocket_data = response.json()

print(f"Total rockets: {len(rocket_data)}")

Total rockets: 4


In [128]:
#2: Convert to DataFrame
rocket_df = pd.DataFrame(rocket_data)

In [129]:
#3: Select required columns
rocket_df = rocket_df[["id", "name", "type", "active", "stages"]]

In [130]:
# Preview data
print(rocket_df.head())
print(rocket_df.info())

                         id          name    type  active  stages
0  5e9d0d95eda69955f709d1eb      Falcon 1  rocket   False       2
1  5e9d0d95eda69973a809d1ec      Falcon 9  rocket    True       2
2  5e9d0d95eda69974db09d1ed  Falcon Heavy  rocket    True       2
3  5e9d0d96eda699382d09d1ee      Starship  rocket   False       2
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      4 non-null      object
 1   name    4 non-null      object
 2   type    4 non-null      object
 3   active  4 non-null      bool  
 4   stages  4 non-null      int64 
dtypes: bool(1), int64(1), object(3)
memory usage: 264.0+ bytes
None


In [131]:
print(rocket_df)

                         id          name    type  active  stages
0  5e9d0d95eda69955f709d1eb      Falcon 1  rocket   False       2
1  5e9d0d95eda69973a809d1ec      Falcon 9  rocket    True       2
2  5e9d0d95eda69974db09d1ed  Falcon Heavy  rocket    True       2
3  5e9d0d96eda699382d09d1ee      Starship  rocket   False       2


## Step 3: Merge Launch and Rocket Data
● Join the two DataFrames on rocket ID using  

In [132]:
# Merge launches and rockets on rocket ID
# df has 'rocket' column (rocket_id) and rocket_df has 'id' column
merged_df = pd.merge(df, rocket_df, left_on='rocket', right_on='id', how="left")

# Display merged data with selected columns
print("Merged Launch and Rocket Data:\n")
print(merged_df.head(15))
print(f"\nTotal merged records: {len(merged_df)}")
print(f"\nMerged DataFrame shape: {merged_df.shape}")

Merged Launch and Rocket Data:

                  name_x                  date_utc success  \
0              FalconSat 2006-03-24 22:30:00+00:00   False   
1                DemoSat 2007-03-21 01:10:00+00:00   False   
2            Trailblazer 2008-08-03 03:34:00+00:00   False   
3                 RatSat 2008-09-28 23:15:00+00:00    True   
4               RazakSat 2009-07-13 03:35:00+00:00    True   
5   Falcon 9 Test Flight 2010-06-04 18:45:00+00:00    True   
6                 COTS 1 2010-12-08 15:43:00+00:00    True   
7                 COTS 2 2012-05-22 07:44:00+00:00    True   
8                  CRS-1 2012-10-08 00:35:00+00:00    True   
9                  CRS-2 2013-03-01 19:10:00+00:00    True   
10              CASSIOPE 2013-09-29 16:00:00+00:00    True   
11                 SES-8 2013-12-03 22:41:00+00:00    True   
12             Thaicom 6 2014-01-06 18:06:00+00:00    True   
13                 CRS-3 2014-04-18 19:25:00+00:00    True   
14        OG-2 Mission 1 2014-07-14 15

## Step 4: Add Simulated Country Information
● Add a new column country and randomly assign one of these values:
['USA', 'Russia', 'India', 'China', 'France']

In [156]:
import numpy as np

In [157]:
# List of countries
countries = ['USA', 'Russia', 'India', 'China', 'France']
np.random.seed(42) # For reproducible results

In [158]:
# Assign random country to each row
merged_df["country"] = np.random.choice(countries, size=len(merged_df))

In [159]:
# Display sample of data with new country column
print("Sample of merged data with country information:\n")
print(merged_df.head(10))

Sample of merged data with country information:

                 name_x                  date_utc success  \
0             FalconSat 2006-03-24 22:30:00+00:00   False   
1               DemoSat 2007-03-21 01:10:00+00:00   False   
2           Trailblazer 2008-08-03 03:34:00+00:00   False   
3                RatSat 2008-09-28 23:15:00+00:00    True   
4              RazakSat 2009-07-13 03:35:00+00:00    True   
5  Falcon 9 Test Flight 2010-06-04 18:45:00+00:00    True   
6                COTS 1 2010-12-08 15:43:00+00:00    True   
7                COTS 2 2012-05-22 07:44:00+00:00    True   
8                 CRS-1 2012-10-08 00:35:00+00:00    True   
9                 CRS-2 2013-03-01 19:10:00+00:00    True   

                                             details  \
0   Engine failure at 33 seconds and loss of vehicle   
1  Successful first stage burn and transition to ...   
2  Residual stage 1 thrust led to collision betwe...   
3  Ratsat was carried to orbit on the first succe...   

In [161]:
print(f"Country distribution:")
print(merged_df['country'].value_counts())

Country distribution:
country
China     50
USA       45
India     38
France    36
Russia    36
Name: count, dtype: int64


## Step 5: Store Merged Data in SQLite3
● Use sqlite3 to create a connection and save the merged DataFrame as a table named
launches

● Table should contain all merged columns including country

In [146]:
import sqlite3

In [147]:
#1: Create connection (this creates file if it doesn't exist)
conn = sqlite3.connect("spacex.db")

In [148]:
#2: Store DataFrame into SQLite table
merged_df.to_sql("launches", conn, if_exists="replace", index=False)

205

In [149]:
#3 Save the merged DataFrame to a table named 'launches'
# Confirm table creation (Verify the data was saved)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM launches")
total_records = cursor.fetchone()[0]
cursor.execute("PRAGMA table_info(launches)")
columns = cursor.fetchall()
column_names = [col[1] for col in columns]

print("Data successfully saved to 'spacex.db' as table 'launches'.")
print(f"Table 'launches' created with {total_records} records")
print(f"Columns: {', '.join(column_names)}")

[('launches',)]
Data successfully saved to 'spacex.db' as table 'launches'.
Table 'launches' created with 205 records
Columns: name_x, date_utc, success, details, rocket, year, id, name_y, type, active, stages, country


In [150]:
# Show sample data from database
cursor.execute("SELECT name_x, year, success, name_y, country FROM launches LIMIT 5")
sample_data = cursor.fetchall()
print(f"\n Sample data from database:")
for row in sample_data:
    print(f"  {row[0]} ({row[1]}) - {row[2]} - {row[3]} - {row[4]}")


 Sample data from database:
  FalconSat (2006) - 0 - Falcon 1 - China
  DemoSat (2007) - 0 - Falcon 1 - France
  Trailblazer (2008) - 0 - Falcon 1 - India
  RatSat (2008) - 1 - Falcon 1 - France
  RazakSat (2009) - 1 - Falcon 1 - France


In [151]:
# Close connection
conn.close()
print(f"Database connection closed")

Database connection closed


## Step 6: Run SQL Queries on the Data to analyze
1. Launches by Country
2. Which year had the highest number of launches?
3. Top 5 Missions by Launch Count

In [152]:
# Connect to the SQLite database
conn = sqlite3.connect('spacex.db')

print(" SpaceX Launch Data Analysis\n")

# 1. Launches by Country
query1 = """
SELECT country, COUNT(*) AS total_launches
FROM launches
GROUP BY country
ORDER BY total_launches DESC;
"""

df1 = pd.read_sql(query1, conn)
print("1. Launches by Country:")
print(df1)

 SpaceX Launch Data Analysis

1. Launches by Country:
  country  total_launches
0   China              50
1     USA              45
2   India              38
3  Russia              36
4  France              36


In [153]:
# 2. Which year had the highest number of launches?
query2 = """
SELECT year, COUNT(*) AS total_launches
FROM launches
GROUP BY year
ORDER BY total_launches DESC
LIMIT 1;
"""

df2 = pd.read_sql(query2, conn)
print("2. Year with the highest number of launches:")
print(df2)

2. Year with the highest number of launches:
   year  total_launches
0  2022              62


In [154]:
# 3. Top 5 Missions by Launch Count
query3 = """
SELECT name_x AS mission_name, COUNT(*) AS launch_count
FROM launches
GROUP BY mission_name
ORDER BY launch_count DESC
LIMIT 5;
"""

df3 = pd.read_sql(query3, conn)
print("3. Top 5 Missions by Launch Count:")
print(df3)

3. Top 5 Missions by Launch Count:
                mission_name  launch_count
0  ispace Mission 1 & Rashid             1
1                       ZUMA             1
2     WorldView Legion 1 & 2             1
3        Viasat-3 & Arcturus             1
4                    USSF-44             1


In [155]:
# Close the connection
conn.close()
print("Database connection closed")

Database connection closed
